# Batch Data Extraction from Images (OCR + Regex + NER Hybrid)

Processes every image in a folder and extracts structured fields from each one.

Pipeline per image:
1. **Preprocessing** — grayscale → 2× resize → denoise → Otsu threshold → deskew
2. **OCR** — Tesseract `image_to_string` on the preprocessed image
3. **Field extraction (hybrid):**
   - **NER** (`Davlan/xlm-roberta-large-ner-hrl`) → `patient_name`, `doctor_name`, `clinic_name`, dates — works on any language/template, no hardcoded phrase patterns
   - **Regex** → `certificate_id` only — alphanumeric codes near a label are not a standard NER entity class
4. **Bounding boxes** — `image_to_data` for word-level coordinates

Results are collected into a summary table at the end.

**Prerequisites:**
- Tesseract binary installed (`brew install tesseract` on macOS)
- `pip install transformers torch` (NER model — downloads ~1.2 GB on first run)

## Step 0 — Environment Setup

Adds the notebook's parent folder to Python's import path so project modules
(`common.*`, `pipeline.*`, etc.) can be imported without installing the package.

- `sys.path.insert(0, ...)` puts this project **before** any similarly-named
  installed package, so local code always wins.
- `Path.cwd().parent` resolves to the notebook's parent directory.
- Prints the resolved project root as a sanity check.


In [ ]:
# Make the project root importable so `common.*` and `pipeline.*` resolve.
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent  # notebook lives in uday/, root is one level up
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

## Configuration & Image Discovery

Defines **which images the notebook will process** and **which one to inspect**
in the single-image cells below.

| Variable | Meaning |
|---|---|
| `IMAGE_FOLDER` | Folder scanned for input images |
| `INSPECT_INDEX` | Index into the sorted file list — controls Step 2 inspection |
| `EXTENSIONS`   | File suffixes accepted (case-insensitive) |

The cell enumerates the folder, filters by extension, sorts the results, and
prints the list with an arrow marker (`← inspect`) next to whichever file will
be used for single-image inspection. Raises `FileNotFoundError` if no matching
files exist so the notebook fails loudly instead of silently doing nothing.


In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
# Root folder scanned recursively — every image/PDF in any subfolder is picked up.
# Notebook lives in <root>/Hugging face model/Davlan ---- xlm-roberta-large-ner-hrl/,
# so we go up two levels to reach the workspace root.
IMAGE_FOLDER = Path.cwd().parent.parent / "test_data" / "sick_notes"  # noqa: PLC0415 --- IGNORE ---

# Which image to use for the single-image inspection cells below.
INSPECT_INDEX = 0

# Supported extensions.
EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".tif", ".pdf"}
# ───────────────────────────────────────────────────────────────────────────────

if not IMAGE_FOLDER.exists():
    raise FileNotFoundError(f"Folder not found: {IMAGE_FOLDER}")

image_paths = sorted(
    p for p in IMAGE_FOLDER.rglob("*")
    if p.is_file() and p.suffix.lower() in EXTENSIONS
)

if not image_paths:
    raise FileNotFoundError(f"No images found under {IMAGE_FOLDER}")

print(f"Found {len(image_paths)} file(s) under {IMAGE_FOLDER}:")
for i, p in enumerate(image_paths):
    marker = " ← inspect" if i == INSPECT_INDEX else ""
    rel = p.relative_to(IMAGE_FOLDER)
    print(f"  [{i}] {rel}{marker}")

## Step 1 — Helper Functions

All processing logic lives in the code cell below so every batch iteration calls
the same functions. This cell defines the OCR configuration, image
preprocessing, OCR wrappers, and the date-parsing helpers documented in the
"Date Parsing" section further down.

### OCR configuration
####  Page Segmentation Mode PSM
- We are using two PSM Mode 3 and 6
- ***--psm 3***: Fully automatic page segmentation. This is the default behavior and acts as a general-purpose mode for standard documents.
- ***--psm 6***: Assumes a single uniform block of text. Best suited for paragraphs or basic document pages.

| Constant | Value | Purpose |
|---|---|---|
| `_OCR_CONFIG_DEFAULT` | `--oem 3 --psm 6` | LSTM engine, **single uniform text block** — default for structured certificates/forms |
| `_OCR_CONFIG_SPARSE`  | `--oem 3 --psm 11` | LSTM engine, **sparse text** — fallback for stamped / scattered layouts |
| `_OCR_LANGUAGES`      | `eng+deu+fra+spa+ita+nld+pol+por+rus+chi_sim` | All 10 languages the NER model supports, joined with `+` so Tesseract loads every model together — matches NER coverage exactly |

**Prerequisite:** `brew install tesseract-lang` (installs every language pack).

### `preprocess(image) → (image, otsu_threshold)`

Turns a raw input image into a clean binary image that Tesseract can OCR reliably:

1. **Grayscale** — `image.convert("L")`.
2. **2× upscale** — LANCZOS resample; small fonts become legible.
3. **Denoise** — 3×3 median filter (removes salt-and-pepper OCR noise).
4. **Otsu binarisation** — inlined pure-NumPy implementation that picks the
   threshold maximising **inter-class variance** across the 256-bin histogram.
   Returns the chosen threshold alongside the image so you can inspect it.
5. **Deskew** — scans rotations from −5° to +5° in 21 steps and picks the angle
   with the highest row-sum variance (sharpest horizontal text lines).

### OCR wrappers

| Function | Returns | Notes |
|---|---|---|
| `ocr_text(preprocessed, config)`     | `str` (full text) | Calls `pytesseract.image_to_string` with all 10 languages |
| `ocr_bbox(preprocessed, config)`     | `list[dict]` (`word`, `x`, `y`, `w`, `h`, `conf`) | Filters out empty words and negative-confidence entries |
| `ocr_with_fallback(preprocessed)`    | `(text, words, config_used)` | Runs PSM 6 first; only retries with PSM 11 if `extract_fields(text)` produces no `patient_name` |
| `draw_bboxes(preprocessed, words)`   | `PIL.Image` | Copy of the image with red rectangles around each recognised word — used for visual OCR-quality checks |

### Date-parsing helpers

Also defined in this cell (documented in detail in the "Date Parsing" section below):

- `_DATE_PATTERNS`, `_YEAR_RE`, `_parse_date`, `find_all_dates` — Layer 1 numeric regex.
- `_extract_document_year` — document-wide fallback year for year-less strings.
- `parse_ner_date_string` — Layer 2 `dateparser` wrapper for NER `DATE` entities.
- `_ISSUE_DATE_RE`, `_find_labelled_issue_date` — label-proximity issue-date lookup.


## Date Parsing — Layered, Cheapest-First

Dates are resolved in two layers. Both operate on the **OCR text string** (not the image) — `dateparser` never touches the image, it works on the same text the regex does.

```
Image
  │
  ▼
Tesseract OCR  →  raw text string
  │
  ▼
extract_fields(text)
  │
  ├─ Layer 1: find_all_dates(text)         ← numeric regex over full text (fast, free)
  │             17/08/2025 · 17.08.2025 · 2025-08-17 · 23.10.02 (DD.MM.YY)
  │
  ├─ Layer 2: parse_ner_date_string(ds)    ← dateparser, ONLY on NER DATE strings
  │             that Layer 1 missed         "30. Mai 1995" · non-Latin scripts
  │
  └─ Issue-date selection
       _find_labelled_issue_date()  ← prefers date on same line as issue/datum label
       fallback: min(valid_dates)   ← earliest date if no label found
```

### Why two layers

| Layer | Handles | Cost | Language |
|---|---|---|---|
| **1 — numeric regex** | `17/08/2025`, `17.08.2025`, `2025-08-17`, `23.10.02` | instant | any (pure digits) |
| **2 — dateparser** | `30. Mai 1995`, `17. August 2025`, non-Latin | small | ~200 langs |

Layer 1 catches the common numeric case at zero cost. Layer 2 only runs on the **DATE strings the NER model already produced** — not the whole document — so `dateparser` cost stays proportional to the number of real dates, not text length.

### 2-digit year handling (DD.MM.YY)

`23.10.02` is matched by the `dmy2` pattern (listed after all 4-digit patterns so `17.08.2025` never hits it). Year expansion uses the ISO 8601 pivot: **YY ≥ 50 → 19YY**, **YY < 50 → 20YY**.

### Issue-date selection — label-proximity first

`min(valid_dates)` picks the earliest date, which for a document containing a patient birthdate (`1995-05-30`) returns the wrong value. `_find_labelled_issue_date()` scans each OCR line for a keyword match first:

```
Keywords: issue date · issued · date of issue · ausstellungsdatum · ausgestellt · datum
```

If a matching line also contains a date, that date is used. `min()` is only the fallback when no labelled line exists.

### The key insight — dateparser needs an isolated substring

`dateparser.parse()` fails on a full paragraph but succeeds on a date-like chunk:

```python
dateparser.parse("AUSSTELLUNGSDATUM: 17. August 2025 DOKUMENT-NR...")   # → None  ❌
dateparser.parse("17. August 2025")                                     # → 2025-08-17  ✅
```

We don't need a separate candidate-hunting regex — the **NER `DATE` entity IS the isolated substring**. NER already extracts `"17. August 2025"` as one entity, so we feed exactly that to `dateparser`.

### Layer 2 loop — single scan per NER string

```python
for ds in date_strings:
    layer1_result = find_all_dates(ds)   # scan once, store result
    if not layer1_result:
        _add(parse_ner_date_string(ds))  # dateparser only if Layer 1 missed it
```

The result is stored in `layer1_result` so `find_all_dates(ds)` is called exactly once per NER string.

### Caveat — OCR noise

Tesseract may misread `17. August 2025` as `17, Auqust 2025`. `dateparser` is fuzzy-tolerant but not perfect, so OCR quality still affects Layer 2 results.


In [ ]:

import re
from datetime import date
from typing import Any

import numpy as np
import pytesseract  # type: ignore[import-untyped]
from PIL import Image, ImageDraw, ImageFilter

# Point pytesseract to the Homebrew-installed Tesseract binary (macOS).
pytesseract.pytesseract.tesseract_cmd = "/opt/homebrew/bin/tesseract"


# ── OCR configs ────────────────────────────────────────────────────────────────
# PSM 6  — single uniform text block; best for structured certificates/forms.
# PSM 11 — sparse text; fallback for stamped / scattered layouts.
_OCR_CONFIG_DEFAULT = r"--oem 3 --psm 6"
_OCR_CONFIG_SPARSE  = r"--oem 3 --psm 11"

# All 10 languages supported by Davlan/xlm-roberta-large-ner-hrl.
# Joined with '+' so Tesseract uses all models together — matches NER coverage exactly.
# Prerequisite: brew install tesseract-lang  (installs all language packs)
_OCR_LANGUAGES = "+".join([
    "eng",      # English
    "deu",      # German
    "fra",      # French
    "spa",      # Spanish
    "ita",      # Italian
    "nld",      # Dutch
    "pol",      # Polish
    "por",      # Portuguese
    "rus",      # Russian
    "chi_sim",  # Chinese (Simplified)
])


# ── Preprocessing ──────────────────────────────────────────────────────────────

def preprocess(image: Image.Image) -> tuple[Image.Image, int]:
    """Return (preprocessed_image, otsu_threshold)."""
    gray = image.convert("L")
    w, h = gray.size
    resized = gray.resize((w * 2, h * 2), Image.LANCZOS)
    denoised = resized.filter(ImageFilter.MedianFilter(size=3))

    arr = np.array(denoised, dtype=np.uint8)
    hist, _ = np.histogram(arr.ravel(), bins=256, range=(0, 256))
    total = arr.size
    sum_total = np.dot(np.arange(256), hist)
    best_thresh, best_var, w_bg, sum_bg = 0, 0.0, 0, 0
    for t in range(256):
        w_bg += hist[t]
        if w_bg == 0:
            continue
        w_fg = total - w_bg
        if w_fg == 0:
            break
        sum_bg += t * hist[t]
        mean_bg = sum_bg / w_bg
        mean_fg = (sum_total - sum_bg) / w_fg
        var = w_bg * w_fg * (mean_bg - mean_fg) ** 2
        if var > best_var:
            best_var, best_thresh = var, t
    binary = Image.fromarray(np.where(arr > best_thresh, 255, 0).astype(np.uint8))

    best_angle, best_score = 0.0, -1.0
    for angle in np.linspace(-5.0, 5.0, 21):
        rotated = binary.rotate(angle, expand=False, fillcolor=255)
        score = float(np.var(np.array(rotated).sum(axis=1).astype(float)))
        if score > best_score:
            best_score, best_angle = score, angle
    preprocessed = binary.rotate(best_angle, expand=True, fillcolor=255)

    return preprocessed, best_thresh


# ── OCR ────────────────────────────────────────────────────────────────────────

def ocr_text(preprocessed: Image.Image, config: str = _OCR_CONFIG_DEFAULT) -> str:
    """Extract all text using all 10 NER-supported languages — no lang hardcoding."""
    return str(pytesseract.image_to_string(preprocessed, lang=_OCR_LANGUAGES, config=config))


def ocr_bbox(preprocessed: Image.Image, config: str = _OCR_CONFIG_DEFAULT) -> list[dict[str, Any]]:
    """Return one dict per recognised word with x/y/w/h/conf."""
    data: dict[str, list[Any]] = pytesseract.image_to_data(
        preprocessed, lang=_OCR_LANGUAGES, output_type=pytesseract.Output.DICT, config=config
    )
    words: list[dict[str, Any]] = []
    for word, x, y, w, h, conf in zip(
        data["text"], data["left"], data["top"],
        data["width"], data["height"], data["conf"],
    ):
        try:
            conf_val = float(conf)
        except (ValueError, TypeError):
            continue
        if not str(word).strip() or conf_val < 0:
            continue
        words.append({"word": word, "x": x, "y": y, "w": w, "h": h, "conf": conf_val})
    return words


def ocr_with_fallback(
    preprocessed: Image.Image,
) -> tuple[str, list[dict[str, Any]], str]:
    """Run OCR with PSM 6; retry with PSM 11 if no patient_name is found.

    Returns (text, words, config_used).
    """
    text  = ocr_text(preprocessed, _OCR_CONFIG_DEFAULT)
    words = ocr_bbox(preprocessed, _OCR_CONFIG_DEFAULT)

    if not extract_fields(text).get("patient_name"):
        sparse_text  = ocr_text(preprocessed, _OCR_CONFIG_SPARSE)
        sparse_words = ocr_bbox(preprocessed, _OCR_CONFIG_SPARSE)
        if extract_fields(sparse_text).get("patient_name"):
            return sparse_text, sparse_words, _OCR_CONFIG_SPARSE

    return text, words, _OCR_CONFIG_DEFAULT


def draw_bboxes(preprocessed: Image.Image, words: list[dict[str, Any]]) -> Image.Image:
    """Return a copy of preprocessed with red bounding boxes drawn over each word."""
    annotated = preprocessed.convert("RGB")
    draw = ImageDraw.Draw(annotated)
    for w in words:
        draw.rectangle(
            (w["x"], w["y"], w["x"] + w["w"], w["y"] + w["h"]),
            outline="red",
            width=2,
        )
    return annotated


# ── Date parsing — layered, cheapest-first ────────────────────────────────────
#
# Layer 1 (fast, free): numeric regex — handles fully-numeric dates including the
#   2-digit-year DD.MM.YY variant common in German documents (e.g. "23.10.02").
#   4-digit patterns are listed first so "17.08.2025" never falls into the YY branch.
# Layer 2 (multilingual): dateparser — runs only on the DATE strings the NER model
#   already produced, to resolve spelled-out / non-Latin dates like "30. Mai 1995"
#   or year-less spellings like "August 19".
# Issue-date selection: label-proximity lookup first (searches for issue/datum/
#   ausgestellt on the same line as a found date), falls back to min() otherwise.

import dateparser  # type: ignore[import-untyped]

_DATE_PATTERNS = [
    # 4-digit year formats (unambiguous) — must appear before the YY pattern.
    (r"\b(\d{2})/(\d{2})/(\d{4})\b", "dmy"),
    (r"\b(\d{2})\.(\d{2})\.(\d{4})\b", "dmy"),
    (r"\b(\d{4})-(\d{2})-(\d{2})\b", "ymd"),
    (r"\b(\d{2})-(\d{2})-(\d{4})\b", "dmy"),
    # 2-digit year — DD.MM.YY (common in German documents).
    # Only reached when no 4-digit pattern matched first.
    (r"\b(\d{2})\.(\d{2})\.(\d{2})\b", "dmy2"),
]

# Any 4-digit year 19xx or 20xx anywhere in the document — used as a fallback
# for year-less NER date strings (e.g. "August 19" in a document that mentions
# "2025" in a header/footer/label).
_YEAR_RE = re.compile(r"\b(19|20)\d{2}\b")


def _parse_date(m: re.Match[str], order: str) -> date | None:
    """Build a date from a regex match given the group order."""
    try:
        g1, g2, g3 = int(m.group(1)), int(m.group(2)), int(m.group(3))
        if order == "ymd":
            return date(g1, g2, g3)
        if order == "dmy":
            return date(g3, g2, g1)
        # dmy2 — 2-digit year pivot: ≥50 → 19xx, <50 → 20xx (ISO 8601 convention)
        year = (1900 + g3) if g3 >= 50 else (2000 + g3)  # noqa: PLR2004
        return date(year, g2, g1)
    except ValueError:
        return None


def find_all_dates(text: str) -> list[date]:
    """Layer 1 — return every valid NUMERIC date found in text across all formats."""
    results: list[date] = []
    for pattern, order in _DATE_PATTERNS:
        for m in re.finditer(pattern, text):
            d = _parse_date(m, order)
            if d is not None:
                results.append(d)
    return results


def _extract_document_year(text: str) -> int | None:
    """Return the first plausible 4-digit year (19xx/20xx) found in the text.

    Used as a fallback year for year-less NER date strings like "August 19".
    """
    m = _YEAR_RE.search(text)
    return int(m.group()) if m else None


def parse_ner_date_string(
    date_string: str,
    fallback_year: int | None = None,
) -> date | None:
    """Layer 2 — resolve one NER-detected DATE string via dateparser (any language).

    dateparser expects an isolated date-like substring, which is exactly what an
    NER DATE entity is. Day-first order matches DD/MM and DD.MM conventions.

    REQUIRE_PARTS forces a real day + month in the parse — this rejects a bare
    "19" being interpreted as year 2019 for input "August 19".

    fallback_year: dateparser silently uses today's year when the input has no
    year (e.g. "August 19"). If the document itself contains a 4-digit year,
    pass it here so the returned date reflects the document's own year context
    instead of today's calendar year.

    Returns None if the string cannot be parsed.
    """
    parsed = dateparser.parse(
        date_string,
        settings={
            "DATE_ORDER":     "DMY",
            "STRICT_PARSING": False,
            "REQUIRE_PARTS":  ["day", "month"],
        },
    )
    if parsed is None:
        return None

    # dateparser silently uses today's year for year-less inputs. If the NER
    # string itself has no 4-digit year but the document does, prefer that year.
    if (
        fallback_year is not None
        and parsed.year == date.today().year
        and not re.search(r"\b\d{4}\b", date_string)
    ):
        parsed = parsed.replace(year=fallback_year)

    return parsed.date()


# Issue-date label keywords in English and German (case-insensitive).
# A date on the same line as any of these is preferred over the min() fallback.
_ISSUE_DATE_RE = re.compile(
    r"(?:issue\s*date|issued|date\s*of\s*issue|ausstellungsdatum|ausgestellt|datum)",
    re.IGNORECASE,
)


def _find_labelled_issue_date(text: str, valid_dates: list[date]) -> date | None:
    """Return the date on the same line as an issue-date label keyword.

    Scans each OCR line for a label match; if the same line also contains a
    date from valid_dates (via Layer 1 re-scan), that date is returned.
    Returns None if no labelled line is found — caller falls back to min().
    """
    for line in text.splitlines():
        if not _ISSUE_DATE_RE.search(line):
            continue
        for d in find_all_dates(line):
            if d in valid_dates:
                return d
    return None


print(f"Helper functions loaded — OCR languages: {_OCR_LANGUAGES}")


## Step 1.5 — NER Model Setup (Davlan/xlm-roberta-large-ner-hrl)

Loaded once and reused for all images. Supports 10 languages including English
and German — no language detection needed, the model handles it automatically.

### Entity → field mapping

| NER label | Maps to field |
|---|---|
| `PER`  | `patient_name` (nearest to "Patient") or `doctor_name` (nearest to "Dr./Arzt") |
| `ORG`  | `clinic_name` |
| `DATE` | `issue_date` + `all_dates_found` |

`certificate_id` is handled by targeted regex — it is not a standard NER entity.

### What the code cell below defines

| Object | Type | Purpose |
|---|---|---|
| `_NER_MODEL`     | `str`                        | Hugging Face model ID (`Davlan/xlm-roberta-large-ner-hrl`) |
| `_ner`           | module-level `object` (`None` initially) | Singleton cache — holds the loaded pipeline after the first call |
| `_get_ner()`     | function → HF pipeline       | Lazy loader: returns the cached pipeline, or builds it on first call |

### Lazy singleton pattern — `_get_ner()`

The NER model is ~1.2 GB and takes 1–2 minutes to load. To avoid paying that
cost more than once per session, the pipeline is stored in a module-level
variable and returned on subsequent calls:

```python
_ner: object = None                     # cache slot

def _get_ner() -> object:
    global _ner
    if _ner is None:                    # first call → build
        _ner = hf_pipeline("ner", model=_NER_MODEL,
                           aggregation_strategy="simple")
    return _ner                         # subsequent calls → reuse
```

- **First call** — downloads the model (if not cached on disk), initialises the
  Transformers pipeline, prints progress, and stores it in `_ner`.
- **Every subsequent call** — returns the already-built pipeline instantly.

Only `extract_fields()` calls `_get_ner()`, so **the model is not loaded until
you actually run an extraction** — importing this cell is free.

### `aggregation_strategy="simple"`

XLM-RoBERTa tokenises words into subword pieces (e.g. `"Klinikum"` → `["Klin",
"##ikum"]`). Without aggregation, the raw NER output would emit one entity per
subword. `aggregation_strategy="simple"` merges adjacent same-label subwords
back into whole words/phrases, so `"Dr. Anna Müller"` comes out as **one** `PER`
entity instead of three.

Other options exist (`"first"`, `"average"`, `"max"`) but `"simple"` is the
right default for clean text extraction where you just want the merged word.

### Model cache location

On first call, the model (~1.2 GB) is automatically downloaded and cached by
Hugging Face:

- **macOS/Linux**: `~/.cache/huggingface/hub/`
- **Windows**: `C:\Users\<username>\.cache\huggingface\hub\`

**Check download location with:**
```bash
ls -lh ~/.cache/huggingface/hub/
```

**Or customise the cache location** (before running the notebook):
```bash
export HF_HOME=/custom/path/to/cache
```


In [ ]:
from transformers import pipeline as hf_pipeline  # type: ignore[import-untyped]

# Loaded once — reused for all extract_fields() calls.
# aggregation_strategy="simple" merges subword tokens into whole words/phrases.
_NER_MODEL = "Davlan/xlm-roberta-large-ner-hrl"
_ner: object = None  # lazy-loaded on first call


def _get_ner() -> object:
    """Return the NER pipeline, loading it on first call."""
    global _ner
    if _ner is None:
        print(f"Loading NER model ({_NER_MODEL}) — first call only, ~1–2 min ...")
        _ner = hf_pipeline(
            "ner",
            model=_NER_MODEL,
            aggregation_strategy="simple",
        )
        print("NER model ready.")
    return _ner


print("NER loader defined — model will download on first extract_fields() call.")


## Step 1.6 — Hybrid Field Extraction (`extract_fields`)

The **core extractor**. Turns the raw OCR text into a dictionary of structured
fields by combining NER (from the loaded pipeline) with a few targeted regexes.

### Field routing

| Field | Source | Notes |
|---|---|---|
| `patient_name`          | NER `PER`             | First non-doctor PER wins |
| `doctor_name`           | NER `PER`             | Detected via `Dr.` / `Or.` / `0r.` prefix pre-scan (OCR-noise-tolerant) |
| `clinic_name`           | NER `ORG`             | Longest ORG match preferred |
| `certificate_id`        | Regex `_CERT_ID_RE`   | Label + alphanumeric code, must contain at least one digit |
| `issue_date`            | Layered date pipeline | Label-proximity first, else earliest date |
| `all_dates_found`       | Layered date pipeline | All numeric + spelled-out dates, ISO 8601 |
| `contains_future_dates` | Derived               | `True` if any parsed date is after today |

### Internal flow

1. Run NER once → collect `PER`, `ORG`, `DATE` entities.
2. Pre-scan text with `_DR_PREFIX_RE` to classify PER entities as doctor vs. patient.
3. **Layer 1 dates:** regex over the full text (numeric formats + `DD.MM.YY`).
4. **Layer 2 dates:** `dateparser` on NER `DATE` strings Layer 1 missed, using a
   document-wide 4-digit year as fallback for year-less strings like `August 19`.
5. Pick `issue_date` — label-proximity first, else `min(valid_dates)`.
6. Regex-extract `certificate_id`.
7. Return a plain `dict[str, Any]` so the batch loop can spread it with `**fields`.


In [ ]:

# ── Hybrid field extraction ────────────────────────────────────────────────────
#
# NER  → patient_name, doctor_name, clinic_name, issue_date, all_dates_found
#          Works on any language and any document template — no format assumptions.
#
# Regex → certificate_id only
#          Alphanumeric codes near a document/certificate label are not a standard
#          NER entity class; targeted regex is simpler and more reliable here.
#
# Dates (layered, cheapest-first):
#   Layer 1  find_all_dates(text)        — numeric regex, including DD.MM.YY
#   Layer 2  parse_ner_date_string(ds)   — dateparser on each NER DATE string
#                                          that Layer 1 did not already cover.
#                                          Year-less strings like "August 19"
#                                          get a fallback year pulled from any
#                                          4-digit year found elsewhere in the
#                                          same document.
# Issue date: _find_labelled_issue_date() looks for a label keyword (issue date/
#   datum/ausgestellt) on the same line as a found date; falls back to min().
# ──────────────────────────────────────────────────────────────────────────────

# Keywords used to assign PER entities to patient role (used as fallback).
# Multilingual: English + German.
_PATIENT_KEYWORDS = {"patient", "mr", "mr.", "mrs", "mrs.", "ms", "ms.", "herr", "frau"}

# Pre-scan pattern: names that follow a doctor prefix anywhere in the document.
# Covers OCR noise variants: "Or." and "0r." are common Tesseract misreads of "Dr."
_DR_PREFIX_RE = re.compile(
    r"\b(?:Dr|Or|0r)\.?\s+([A-Z][a-zA-Z]+(?:\s+[A-Z][a-zA-Z]+)*)",
    re.IGNORECASE,
)

# Regex for certificate-style codes.
# "certificate" alone (without id/no/nr) is NOT matched — avoids capturing decorative
# lines like "SICK LEAVE CERTIFICATE\nCeeeeneeee..." that follow bare CERTIFICATE labels.
# Captured value must be ≤ 25 chars and contain at least one digit (real cert IDs always do).
_CERT_ID_RE = re.compile(
    r"(?:certificate\s+(?:id|no\.?|nr)|document\s*no\.?|patient\s*id|cert\.?\s*no\.?)"
    r"\s*[:\s#]*([A-Z][A-Z0-9]{5,24})",
    re.IGNORECASE,
)


def _classify_per(word: str, context: str, known_doctors: set[str]) -> str:
    """Return 'doctor' or 'patient'.

    Priority order:
    1. Name appeared after a Dr/Or/0r prefix anywhere in the document → doctor.
    2. Surrounding context contains a patient-role keyword → patient.
    3. Default → patient.
    """
    if word in known_doctors:
        return "doctor"
    tokens = set(re.findall(r"\w+\.?", context.lower()))
    if tokens & _PATIENT_KEYWORDS:
        return "patient"
    return "patient"


def extract_fields(text: str) -> dict[str, Any]:
    """Extract structured fields using NER (names/org/dates) + regex (cert ID).

    NER handles any language and template without hardcoded phrase patterns.
    Regex covers certificate_id, which is not a standard NER entity class.
    """
    ner = _get_ner()
    entities: list[dict] = ner(text)  # type: ignore[operator]

    # ── Pre-scan: collect names that follow a doctor prefix (Dr./Or./0r.) ─────
    # This handles OCR misreads of "Dr." before running NER classification.
    known_doctors: set[str] = {
        m.group(1).strip()
        for m in _DR_PREFIX_RE.finditer(text)
    }

    patient_names: list[str] = []
    doctor_names:  list[str] = []
    clinic_names:  list[str] = []
    date_strings:  list[str] = []

    for ent in entities:
        label = ent.get("entity_group", "")
        word  = str(ent.get("word", "")).strip()
        start = int(ent.get("start", 0))

        # Skip noise: single characters or 2-char fragments are never real names/orgs.
        if len(word) < 3:  # noqa: PLR2004
            continue

        if label == "PER":
            lo = max(0, start - 120)  # noqa: PLR2004
            hi = min(len(text), start + 120)  # noqa: PLR2004
            context = text[lo:hi]
            role = _classify_per(word, context, known_doctors)
            if role == "doctor":
                doctor_names.append(word)
            else:
                patient_names.append(word)

        elif label == "ORG":
            clinic_names.append(word)

        elif label == "DATE":
            date_strings.append(word)

    # ── Resolve to single best value per field ─────────────────────────────────
    # First PER entity per role wins; for clinic, prefer the longest ORG match.
    patient_name = patient_names[0] if patient_names else None
    doctor_name  = doctor_names[0]  if doctor_names  else None
    clinic_name  = max(clinic_names, key=len) if clinic_names else None

    # ── Dates: layered, cheapest-first ─────────────────────────────────────────
    # Layer 1 — numeric regex over the full OCR text (covers DD/MM/YYYY, DD.MM.YYYY,
    #           YYYY-MM-DD, DD-MM-YYYY, and the 2-digit-year DD.MM.YY variant).
    # Layer 2 — dateparser on each NER DATE string that Layer 1 did not cover
    #           (spelled-out months like "30. Mai 1995", "August 19", non-Latin).
    valid_dates: list[date] = []
    seen: set[date] = set()

    def _add(d: date | None) -> None:
        """Append a parsed date once, preserving first-seen order."""
        if d is not None and d not in seen:
            seen.add(d)
            valid_dates.append(d)

    # Layer 1
    for d in find_all_dates(text):
        _add(d)

    # Fallback year for Layer 2: prefer a year Layer 1 already parsed, otherwise
    # pull any 19xx/20xx literal from the raw text. This keeps year-less NER
    # strings like "August 19" anchored to the document instead of today's year.
    fallback_year: int | None = (
        max(d.year for d in valid_dates) if valid_dates
        else _extract_document_year(text)
    )

    # Layer 2 — only for NER DATE strings that Layer 1 could not parse.
    # Calling find_all_dates(ds) on the short NER string is cheap; if it returns
    # anything, Layer 1 already covered that date so we skip dateparser entirely.
    for ds in date_strings:
        layer1_result = find_all_dates(ds)
        if not layer1_result:
            _add(parse_ner_date_string(ds, fallback_year=fallback_year))

    # ── Issue date — label-proximity first, min() fallback ─────────────────────
    # Prefer a date that appears on the same line as an issue/datum label keyword.
    # Fall back to the earliest date only if no labelled date is found — this avoids
    # selecting a patient birthdate when the document contains historical dates.
    issue_date: str | None = None
    if valid_dates:
        labelled = _find_labelled_issue_date(text, valid_dates)
        best = labelled if labelled is not None else min(valid_dates)
        issue_date = best.strftime("%d/%m/%Y")

    today = date.today()

    # ── certificate_id via regex ───────────────────────────────────────────────
    # "CERTIFICATE" alone is excluded — label must include ID/NO/NR.
    # Captured value must contain at least one digit (real cert IDs always do).
    certificate_id: str | None = None
    cert_match = _CERT_ID_RE.search(text)
    if cert_match and re.search(r"\d", cert_match.group(1)):
        certificate_id = cert_match.group(1).strip()

    return {
        "patient_name":          patient_name,
        "doctor_name":           doctor_name,
        "clinic_name":           clinic_name,
        "certificate_id":        certificate_id,
        "issue_date":            issue_date,
        "all_dates_found":       [d.isoformat() for d in valid_dates],
        "contains_future_dates": any(d > today for d in valid_dates),
    }


print("Hybrid extract_fields() loaded — NER + regex + layered dates (v5: year-less date support, e.g. 'August 19').")


## PDF → Image Helper (`pdf_page_to_image`)

Renders a single PDF page to a PIL `Image` using **PyMuPDF (`fitz`)** so the
rest of the pipeline (Tesseract, NER, heatmap) can treat PDFs and raster images
uniformly.

- `dpi=200` upscales the render enough for reliable OCR on standard document layouts.
- `alpha=False` produces an opaque RGB pixmap (no transparency channel).
- Renders only the requested page (`page_index=0` by default) — multi-page PDFs
  would need a loop around this helper.

Called from the batch loop when a file's suffix is `.pdf`.


In [ ]:
import fitz  # PyMuPDF


def pdf_page_to_image(pdf_path: Path, page_index: int = 0, dpi: int = 200) -> Image.Image:
    """Render one PDF page to a PIL Image at the given DPI."""
    doc = fitz.open(str(pdf_path))
    page = doc[page_index]
    mat = fitz.Matrix(dpi / 72, dpi / 72)
    pix = page.get_pixmap(matrix=mat, alpha=False)
    return Image.frombytes("RGB", (pix.width, pix.height), pix.samples)


print("PDF helper loaded.")


## Step 2 — Inspect a Single Image

The next few cells run the **full pipeline on just one image** so you can
step through each stage and see intermediate results. This is where you would
debug OCR quality, field extraction, or heatmap output before committing to a
full batch run.

The image used is `image_paths[INSPECT_INDEX]` — change `INSPECT_INDEX` in the
config cell to pick a different file.

**Sub-steps below:**

1. Open the raw image (this cell — displays it inline for a visual reference).
2. Preprocess & deskew.
3. Run OCR with PSM fallback + draw word bounding boxes.
4. Save the word list to JSON.
5. Print raw OCR text + extracted fields.

All intermediate variables are prefixed `inspect_` (e.g. `inspect_text`,
`inspect_words`) so the later heatmap cells can reuse them without recomputing.


In [ ]:
inspect_path = image_paths[INSPECT_INDEX]
inspect_image = Image.open(inspect_path)
print(f"[{INSPECT_INDEX}] {inspect_path.name}  —  size={inspect_image.size}  mode={inspect_image.mode}")
inspect_image

### Preprocess & Deskew the Inspection Image

Runs the `preprocess()` helper on the loaded inspection image and prints the
original vs. preprocessed sizes plus the auto-selected Otsu threshold.

Preprocessing steps applied inside `preprocess()`:

1. Grayscale conversion.
2. 2× upscale (LANCZOS) — helps Tesseract on small fonts.
3. Median filter denoise (kernel size 3).
4. Otsu binarisation — auto-picks a threshold that maximises inter-class variance.
5. Deskew — tests rotations from −5° to +5° in 0.5° steps and picks the angle
   that maximises row-sum variance (sharpest horizontal text lines).

Displaying `inspect_preprocessed` at the end renders the cleaned binary image inline.


In [ ]:
inspect_preprocessed, inspect_thresh = preprocess(inspect_image)
print(f"Original   : {inspect_image.size}")
print(f"Preprocessed: {inspect_preprocessed.size}  (Otsu threshold={inspect_thresh})")
inspect_preprocessed

### Run OCR (with PSM Fallback) + Draw Bounding Boxes

Calls `ocr_with_fallback()`, which:

1. Runs Tesseract with **PSM 6** (uniform text block — default for structured certificates).
2. If no `patient_name` can be extracted from that text, retries with **PSM 11**
   (sparse text — better for stamped / scattered layouts) and keeps whichever
   result actually yields a patient name.

Returns `(text, words, config_used)`:

- `inspect_text` — full OCR text used by the NER model.
- `inspect_words` — list of dicts (`word`, `x`, `y`, `w`, `h`, `conf`) used for
  bounding-box lookups in the heatmap step.
- `inspect_ocr_config` — the PSM string that actually produced usable results,
  printed for transparency.

The cell prints the first 100 words with their coordinates and confidence, then
calls `draw_bboxes()` to render the preprocessed image with red rectangles
around every recognised word so you can visually verify OCR quality.


In [ ]:
# Run OCR with PSM fallback — text, words, and the config actually used are
# all produced here so downstream cells can reference inspect_words directly.
inspect_text, inspect_words, inspect_ocr_config = ocr_with_fallback(inspect_preprocessed)

print(f"OCR config used : {inspect_ocr_config}")
print(f"{len(inspect_words)} words recognised\n")
for w in inspect_words[:100]:
    print(f"{w['word']:20s}  x={w['x']:>4} y={w['y']:>4} w={w['w']:>4} h={w['h']:>4} conf={w['conf']:.1f}")

draw_bboxes(inspect_preprocessed, inspect_words)


### Save Bounding-Box Words to JSON

Dumps the per-word OCR data (position, size, confidence) for the inspection
image to `heatmaps/bbox_words_<stem>.json`.

Useful for:

- Offline inspection of Tesseract output.
- Feeding a downstream tool that needs word-level geometry without re-running OCR.
- Diffing OCR results between runs to catch regressions.

Creates the `heatmaps/` folder if missing and writes UTF-8 JSON with non-ASCII
characters preserved (`ensure_ascii=False`).


In [ ]:
import json

# Save the OCR word list (with bounding boxes + confidence) as JSON.
_words_out = Path.cwd() /"heatmaps"/f"bbox_words_{inspect_path.stem}.json"
_words_out.parent.mkdir(parents=True, exist_ok=True)
with _words_out.open("w", encoding="utf-8") as fh:
    json.dump(inspect_words, fh, indent=2, ensure_ascii=False, default=str)

print(f"Saved {len(inspect_words)} words → {_words_out}")

### Show Raw OCR Text + Extracted Fields (Inspection)

End-to-end sanity check for a single image:

1. Prints the **full OCR text** exactly as returned by Tesseract, so you can
   spot OCR noise directly (e.g. `Or.` where `Dr.` was expected, or a
   misread date character).
2. Runs `extract_fields()` on that text and prints each field with its extracted value.

Use this cell to debug an empty field — the usual causes are:

- OCR misread the label (fix: check preprocessing / language pack).
- The value sits outside the NER-supported entity classes (fix: adjust regex).
- The `Dr.` prefix was not detected (fix: extend `_DR_PREFIX_RE` variants).


In [ ]:
print(f"OCR config used: {inspect_ocr_config}\n")
print("=== Raw OCR text ===")
print(inspect_text)

print("\n=== Extracted fields ===")
for key, value in extract_fields(inspect_text).items():
    print(f"{key:30s}: {value}")


## Step 3 — Batch Processing

Runs the full pipeline on **every image in `IMAGE_FOLDER`** and collects the
results into a list of dicts (which the next cell turns into a DataFrame).

For each file:

1. Load — `pdf_page_to_image()` for `.pdf`, otherwise `Image.open()`.
2. `preprocess()` → cleaned binary image + Otsu threshold.
3. `ocr_with_fallback()` → text, word list, and the PSM config that succeeded.
4. `extract_fields()` → structured dictionary.
5. Append a merged row: `{file, ocr_config, words_detected, otsu_threshold, **fields}`.

The `except Exception` around each iteration is intentional and annotated with
`# noqa: BLE001` — the batch loop must keep going even if a single image
fails, and the failure row records the error string so you can spot it in the
DataFrame.


In [ ]:
results = []

for path in image_paths:
    print(f"Processing {path.name} ...", end=" ", flush=True)
    try:
        if path.suffix.lower() == ".pdf":
            img = pdf_page_to_image(path)
        else:
            img = Image.open(path)
        preprocessed, thresh = preprocess(img)
        text, words, ocr_config = ocr_with_fallback(preprocessed)
        fields = extract_fields(text)

        results.append({
            "file":           path.name,
            "ocr_config":     ocr_config,
            "words_detected": len(words),
            "otsu_threshold": thresh,
            **fields,
        })
        print(f"OK  ({len(words)} words, {ocr_config})")
    except Exception as exc:  # noqa: BLE001  # reason: batch loop must continue on per-image failures
        print(f"FAILED — {exc}")
        results.append({"file": path.name, "error": str(exc)})

print(f"\nDone. {len(results)} image(s) processed.")


### Results as a Pandas DataFrame

Prints the raw `results` list as pretty JSON (for a full, uncollapsed view of
every field, including nested lists like `all_dates_found`), then builds a
`pandas.DataFrame` with a stable, human-friendly column order:

- **Priority columns first** — `file`, then the main extracted fields
  (`patient_name`, `doctor_name`, `clinic_name`, `certificate_id`,
  `issue_date`, `contains_future_dates`, `all_dates_found`), then diagnostic
  columns (`words_detected`, `otsu_threshold`).
- **Any extra columns** (e.g. an `error` field on failed rows) are appended after.

To add a new field to the table later, insert its key in `priority_cols`.


In [ ]:
import pandas as pd
import json

print(json.dumps(results, indent=2, default=str))

df = pd.DataFrame(results)

# Show key fields first, then the rest.
priority_cols = ["file", "patient_name", "doctor_name", "clinic_name",
                 "certificate_id", "issue_date", "contains_future_dates",
                 "all_dates_found", "words_detected", "otsu_threshold"]
ordered_cols = [c for c in priority_cols if c in df.columns] + \
               [c for c in df.columns if c not in priority_cols]

pd.set_option("display.max_colwidth", 40)
pd.set_option("display.max_columns", 20)
df[ordered_cols]


## Step 4 — Field Repetition Analysis + Heatmap

For each extracted field value (patient name, doctor name, dates, etc.) we:

1. **Tokenise** the value into individual words
2. **Scan** the bounding-box word list for case-insensitive matches
3. **Count** how many times each token appears and where (x, y, w, h)
4. **Render** a Gaussian heatmap on the original image — brighter = more occurrences

This shows whether key values (e.g. a name or date) appear multiple times across the
document, which can indicate copy-paste artefacts or repeated stamps.

In [ ]:
import re
from collections import defaultdict


def _clean_word(w: str) -> str:
    """Strip punctuation and lowercase — used for case-insensitive phrase matching."""
    return re.sub(r"[^\w]", "", w).lower()


def _find_phrase(field: str, value: str, words: list[dict]) -> list[dict]:
    """Slide a window over the word list for a consecutive phrase match.

    Returns one hit dict per occurrence with merged bounding box.
    """
    tokens = [_clean_word(t) for t in re.split(r"[\s\-/.,]+", value)
              if re.sub(r"[^\w]", "", t)]
    n = len(tokens)
    if n == 0:
        return []

    cleaned = [_clean_word(w["word"]) for w in words]
    hits: list[dict] = []

    for i in range(len(words) - n + 1):
        if cleaned[i : i + n] == tokens:
            span = words[i : i + n]
            x1 = min(w["x"] for w in span)
            y1 = min(w["y"] for w in span)
            x2 = max(w["x"] + w["w"] for w in span)
            y2 = max(w["y"] + w["h"] for w in span)
            hits.append({
                "x":              x1,
                "y":              y1,
                "w":              x2 - x1,
                "h":              y2 - y1,
                "conf":           sum(w["conf"] for w in span) / n,
                "word":           value,
                "_matched_value": value,
                "_field":         field,
            })

    return hits


def find_field_occurrences(fields: dict, words: list[dict]) -> dict[str, list[dict]]:
    """Return field → list of hit dicts, one per full-phrase occurrence."""
    searchable_fields = [
        "patient_name", "doctor_name", "clinic_name",
        "certificate_id", "issue_date",
    ]

    hits: dict[str, list[dict]] = {}

    for field in searchable_fields:
        value = fields.get(field)
        if value and isinstance(value, str):
            clean_value = value.split("\n")[0].strip()
            if clean_value:
                found = _find_phrase(field, clean_value, words)
                if found:
                    hits[field] = found

    date_hits: list[dict] = []
    for iso_date in (fields.get("all_dates_found") or []):
        dot_date = iso_date[8:10] + "." + iso_date[5:7] + "." + iso_date[0:4]
        for fmt in (iso_date, dot_date):
            date_hits.extend(_find_phrase("all_dates_found", fmt, words))
    if date_hits:
        hits["all_dates_found"] = date_hits

    return hits


def print_occurrence_summary(hits: dict[str, list[dict]]) -> None:
    """Print a per-field occurrence table."""
    if not hits:
        print("No field values found in bounding boxes.")
        return

    for field, matches in hits.items():
        phrase_counts: dict[str, int] = defaultdict(int)
        for m in matches:
            phrase_counts[m["_matched_value"]] += 1

        print(f"\n{'─' * 72}")
        print(f"  Field : {field}")
        for phrase, count in sorted(phrase_counts.items(), key=lambda kv: -kv[1]):
            print(f"  Phrase: '{phrase}'  →  {count} occurrence(s)")

        print(f"  {'phrase':30s}  {'x':>5} {'y':>5}  {'conf':>5}")
        for m in matches:
            print(
                f"  {m['_matched_value']:30s}  {m['x']:>5} {m['y']:>5}  "
                f"{m['conf']:>5.1f}"
            )


# ── Run on the inspect image ────────────────────────────────────────────────────
inspect_fields = extract_fields(inspect_text)
inspect_hits   = find_field_occurrences(inspect_fields, inspect_words)

print_occurrence_summary(inspect_hits)


## Step 4.1 — Heatmap & Bar Chart of Field Occurrences

Renders two visualisations for the inspection image and saves both to
`heatmaps/`:

### `show_heatmap(...)`

Overlays a Gaussian glow on the original image, one colour per field, plus
labelled bounding boxes. Fields that were extracted from text but could not be
matched to a bounding box (e.g. OCR confidence too low, or value spans lines)
are listed in a separate info panel below the image marked *"not located in
image"*.

- `_FIELD_COLOURS` — distinct RGB per field (blue = patient, green = doctor,
  orange = clinic, purple = certificate ID, red = issue date, yellow = all dates).
- `_build_channels(...)` — one Gaussian-blurred heat channel per field, later
  blended additively.
- `sigma=30` controls how wide each hotspot spreads; increase for softer, more
  overlapping halos.

### `show_barchart(...)`

Horizontal bar chart of occurrence counts. Located fields are drawn in their
field colour; unlocated fields appear as grey bars with count `0` and the
label `not located`. Provides a quick visual answer to *"which fields
repeat?"* — high repetition can hint at copy-paste artefacts or repeated
stamps.

Both figures are saved as PNGs (`heatmap_<stem>.png`, `barchart_<stem>.png`)
in the `heatmaps/` folder next to the JSON word dump saved earlier.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from PIL import Image
from scipy.ndimage import gaussian_filter

# Distinct colours per field (RGB 0-1 for matplotlib).
_FIELD_COLOURS = {
    "patient_name":     (0.20, 0.60, 1.00),   # blue
    "doctor_name":      (0.00, 0.78, 0.42),   # green
    "clinic_name":      (1.00, 0.55, 0.00),   # orange
    "certificate_id":   (0.80, 0.20, 0.80),   # purple
    "issue_date":       (1.00, 0.20, 0.20),   # red
    "all_dates_found":  (1.00, 0.85, 0.00),   # yellow
}
_FALLBACK_COLOUR = (0.50, 0.50, 0.50)


def _build_channels(
    hits: dict,
    img_w: int,
    img_h: int,
    scale: float,
    sigma: int,
) -> dict[str, np.ndarray]:
    """Compute one Gaussian heat channel per field."""
    channels: dict[str, np.ndarray] = {}
    for field, matches in hits.items():
        canvas = np.zeros((img_h, img_w), dtype=np.float32)
        for m in matches:
            cx = int((m["x"] + m["w"] / 2) * scale)
            cy = int((m["y"] + m["h"] / 2) * scale)
            cx = min(max(cx, 0), img_w - 1)
            cy = min(max(cy, 0), img_h - 1)
            canvas[cy, cx] += 1.0
        channels[field] = gaussian_filter(canvas, sigma=sigma)
    return channels


def show_heatmap(
    image: Image.Image,
    hits: dict[str, list[dict]],
    extracted_fields: 'dict | None' = None,
    sigma: int = 30,
    heatmap_alpha: float = 0.55,
    save_dir: 'Path | None' = None,
    stem: str = "image",
) -> None:
    """Heatmap overlay showing all extracted fields — located and unlocated.

    Located fields (found in bbox):
      Coloured bounding boxes on the image, Gaussian glow overlay.
      Label shows value + occurrence count.

    Unlocated fields (extracted but not matched to any bbox):
      Listed in a summary panel below the image.
      Shown in grey — they were extracted from text but couldn't be pinned
      to a position (e.g. OCR confidence too low, or field not in bbox list).

    Parameters
    ----------
    hits             : output of find_field_occurrences()
    extracted_fields : output of extract_fields() — used to show unlocated fields
    """
    all_located = list(hits.keys())

    # ── identify unlocated fields ─────────────────────────────────────────────
    unlocated: dict[str, str] = {}
    if extracted_fields:
        searchable = ["patient_name", "doctor_name", "clinic_name",
                      "certificate_id", "issue_date", "all_dates_found"]
        for field in searchable:
            val = extracted_fields.get(field)
            if field == "all_dates_found":
                # Show as unlocated if dates were found but none matched bbox
                if val and field not in hits:
                    unlocated[field] = ", ".join(val)
            elif val and isinstance(val, str):
                clean = val.split("\n")[0].strip()
                if clean and field not in hits:
                    unlocated[field] = clean

    if not all_located and not unlocated:
        print("No field values found in bounding boxes.")
        return

    img_w, img_h = image.size
    scale = 0.5

    # ── layout: image axes + optional unlocated panel ────────────────────────
    has_unlocated = bool(unlocated)
    if has_unlocated:
        fig = plt.figure(figsize=(12, 9))
        ax      = fig.add_axes([0.0, 0.18, 1.0, 0.80])   # image
        ax_info = fig.add_axes([0.0, 0.00, 1.0, 0.17])   # info strip
        ax_info.axis("off")
    else:
        fig, ax = plt.subplots(figsize=(12, 8))

    # ── Gaussian heat overlay ─────────────────────────────────────────────────
    if all_located:
        channels = _build_channels(hits, img_w, img_h, scale, sigma)
        overlay_rgb   = np.zeros((img_h, img_w, 3), dtype=np.float32)
        overlay_alpha = np.zeros((img_h, img_w),    dtype=np.float32)
        for field, channel in channels.items():
            if channel.max() == 0:
                continue
            norm   = channel / channel.max()
            colour = _FIELD_COLOURS.get(field, _FALLBACK_COLOUR)
            for c_idx, c_val in enumerate(colour):
                overlay_rgb[:, :, c_idx] += norm * c_val
            overlay_alpha = np.maximum(overlay_alpha, norm)
        overlay_rgb = np.clip(overlay_rgb, 0, 1)
        ax.imshow(image)
        ax.imshow(
            np.dstack([overlay_rgb, overlay_alpha * heatmap_alpha]),
            interpolation="bilinear",
        )
    else:
        ax.imshow(image)

    # ── bounding boxes for located fields ─────────────────────────────────────
    for field, matches in hits.items():
        colour = _FIELD_COLOURS.get(field, _FALLBACK_COLOUR)
        count  = len(matches)
        for m in matches:
            x, y = m["x"] * scale, m["y"] * scale
            w, h = m["w"] * scale, m["h"] * scale
            ax.add_patch(plt.Rectangle(
                (x, y), w, h, linewidth=1.5, edgecolor=colour, facecolor="none",
            ))
            repeat_tag = f"  ×{count}" if count > 1 else ""
            ax.text(x, y - 3, f"{m['_matched_value']}{repeat_tag}",
                    fontsize=6, color=colour, fontweight="bold", clip_on=True)

    # ── legend for located fields ─────────────────────────────────────────────
    legend_patches = [
        mpatches.Patch(
            color=_FIELD_COLOURS.get(f, _FALLBACK_COLOUR),
            label=f"{f}  (×{len(hits[f])}{'  repeated' if len(hits[f]) > 1 else ''})",
        )
        for f in all_located
    ]
    if unlocated:
        legend_patches.append(
            mpatches.Patch(color=_FALLBACK_COLOUR, label="— not located in image (see below)")
        )
    if legend_patches:
        ax.legend(handles=legend_patches, loc="upper right", fontsize=8, framealpha=0.85)

    ax.set_title("Field occurrences — heatmap overlay", fontsize=12)
    ax.axis("off")

    # ── unlocated fields panel ────────────────────────────────────────────────
    if has_unlocated:
        lines = ["Extracted but not located in image:"]
        for field, val in unlocated.items():
            lines.append(f"   {field}: {val}")
        ax_info.text(
            0.01, 0.85, "\n".join(lines),
            transform=ax_info.transAxes,
            fontsize=9, verticalalignment="top", family="monospace",
            bbox=dict(boxstyle="round,pad=0.4", facecolor="#f5f5f5",
                      edgecolor="#aaaaaa", alpha=0.9),
        )

    fig.tight_layout(rect=[0, 0.17 if has_unlocated else 0, 1, 1])

    if save_dir:
        save_dir.mkdir(parents=True, exist_ok=True)
        out = save_dir / f"heatmap_{stem}.png"
        fig.savefig(out, dpi=150, bbox_inches="tight")
        print(f"Saved heatmap  → {out}")

    plt.show()


def show_barchart(
    hits: dict[str, list[dict]],
    extracted_fields: 'dict | None' = None,
    save_dir: 'Path | None' = None,
    stem: str = "image",
) -> None:
    """Bar chart of occurrence counts — located fields as bars, unlocated as a note.

    Located fields (found in bbox) are shown as coloured bars.
    Unlocated fields (extracted but not matched) are listed as grey bars with count 0.
    """
    # Merge located + unlocated into one ordered list.
    located_fields = list(hits.keys())

    unlocated_fields: list[str] = []
    if extracted_fields:
        searchable = ["patient_name", "doctor_name", "clinic_name",
                      "certificate_id", "issue_date", "all_dates_found"]
        for field in searchable:
            val = extracted_fields.get(field)
            if field == "all_dates_found":
                if val and field not in hits:
                    unlocated_fields.append(field)
            elif val and isinstance(val, str) and val.split("\n")[0].strip():
                if field not in hits:
                    unlocated_fields.append(field)

    all_fields  = located_fields + unlocated_fields
    if not all_fields:
        print("No fields to display.")
        return

    counts      = [len(hits.get(f, [])) for f in all_fields]
    bar_colours = [
        _FIELD_COLOURS.get(f, _FALLBACK_COLOUR) if f in hits else (0.75, 0.75, 0.75)
        for f in all_fields
    ]

    labels = []
    for f in all_fields:
        if f in hits:
            phrase = hits[f][0]["_matched_value"] if hits[f] else ""
            rep    = "  repeated" if len(hits[f]) > 1 else ""
            labels.append(f"{f}\n'{phrase}'  (×{len(hits[f])}){rep}")
        else:
            val = extracted_fields.get(f, "") if extracted_fields else ""
            if isinstance(val, list):
                val = ", ".join(val)
            clean = str(val).split("\n")[0].strip() if val else "—"
            labels.append(f"{f}  [not located]\n'{clean}'")

    fig, ax = plt.subplots(figsize=(9, max(3, len(all_fields) * 1.1)))
    bars = ax.barh(labels, counts, color=bar_colours, edgecolor="white", height=0.5)
    ax.bar_label(bars, padding=4, fontsize=10,
                 labels=[str(c) if c > 0 else "not located" for c in counts])
    ax.set_xlabel("Number of full-phrase occurrences in image", fontsize=10)
    ax.set_title("Occurrence count per extracted field", fontsize=12)
    ax.invert_yaxis()
    ax.set_xlim(0, max(counts + [1]) + 2)
    fig.tight_layout()

    if save_dir:
        save_dir.mkdir(parents=True, exist_ok=True)
        out = save_dir / f"barchart_{stem}.png"
        fig.savefig(out, dpi=150, bbox_inches="tight")
        print(f"Saved bar chart → {out}")

    plt.show()


# ── Render both figures for the inspect image ────────────────────────────────
_save_dir = Path.cwd() / "heatmaps"

show_heatmap(
    inspect_image, inspect_hits,
    extracted_fields=inspect_fields,
    save_dir=_save_dir, stem=inspect_path.stem,
)
show_barchart(
    inspect_hits,
    extracted_fields=inspect_fields,
    save_dir=_save_dir, stem=inspect_path.stem,
)
